In [ ]:
"""
Script: evaluate_4foil_classifier.py

Description:
    Loads 4-foil knot 3D dot data from CSV files, constructs a PyTorch DataLoader,
    initializes a pre-trained 3D CNN classifier (Classifier3D),
    runs inference on the test dataset, and visualizes the confusion matrix.

Sections:
    1. Imports and device configuration
    2. Hyperparameter and resolution settings
    3. Knot definitions and data folders
    4. Data loading: building 3D dot arrays
    5. Tensor conversion and DataLoader creation
    6. Model loading and setup
    7. Inference and confusion matrix plotting
"""

In [ ]:
# -----------------------------------------------------------------------------
# 1. Imports and Device Configuration
# -----------------------------------------------------------------------------
import sys
sys.path.append('../')  # Include parent directory for custom packages
import json
import csv

import itertools
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix
import seaborn as sns

from classifier_models import Classifier3D
from extra_functions_package.all_knots_functions import *  # Custom knot utilities

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# -----------------------------------------------------------------------------
# 2. Hyperparameter and Resolution Settings
# -----------------------------------------------------------------------------
hyperparams = {
    'learning_rate': 1e-5,  # (unused here) optimizer LR
    'patience': 5,          # (unused here) LR scheduler patience
    'factor': 0.2,          # (unused here) LR decay factor
    'batch_size': 64        # Batch size for inference
}

# Target spatial resolution for 3D dot grids
desired_res = (32, 32, 32)

In [ ]:
# -----------------------------------------------------------------------------
# 3. Knot Definitions and Data Folders
# -----------------------------------------------------------------------------
# Generate all 4-foil knot labels (e.g., '0000', '0001', ..., '2222')
foils = list(itertools.product(range(3), repeat=4))
knots = [''.join(map(str, foil)) for foil in foils]
# Map each knot string to its integer class label
knot_types = {knot: idx for idx, knot in enumerate(knots)}

# List of folders containing CSV data files
folders = [
    '../4foils_L270_0.05_1_64x64x64_v1',
]
num_classes = len(knots)

# Increase CSV parser field size for large JSON entries
csv.field_size_limit(10_000_000)


In [ ]:
# -----------------------------------------------------------------------------
# 4. Data Loading: Building 3D Dot Arrays
# -----------------------------------------------------------------------------
X_list, Y_list = [], []
flag_print_shape = True  # Only print grid shape for the first sample

for folder in folders:
    for knot in knots:
        filename = f"{folder}/data_{knot}.csv"
        try:
            with open(filename, 'r') as file:
                reader = csv.reader(file)
                for row in reader:
                    # Each row is a JSON string: [metadata, (Nx,Ny,Nz), point coords...]
                    data_list = json.loads(row[0])
                    data_array = np.array(data_list, dtype=object)

                    # Extract grid dimensions and the list of points
                    _, (Nx, Ny, Nz), *points = data_array
                    points = np.array(points, dtype=int)

                    # Print the original shape once
                    if flag_print_shape:
                        print(f"Original grid shape: ({Nx}, {Ny}, {Nz})")
                        flag_print_shape = False

                    # Rescale coordinates if the grid size differs from desired_res
                    if (Nx, Ny, Nz) != desired_res:
                        scale = np.array(desired_res) / np.array([Nx, Ny, Nz])
                        points = np.rint(points * scale).astype(int)

                    # Create binary 3D voxel grid of knot points
                    voxels = np.zeros(desired_res, dtype=int)
                    for x, y, z in points:
                        if 0 <= x < desired_res[0] and 0 <= y < desired_res[1] and 0 <= z < desired_res[2]:
                            voxels[x, y, z] = 1

                    # Store sample and its class label
                    X_list.append(voxels)
                    Y_list.append(knot_types[knot])

        except FileNotFoundError:
            print(f"File not found: {filename}")
        except json.JSONDecodeError:
            print(f"Error decoding JSON in file: {filename}")

print(f"Loaded total samples: {len(X_list)} ({len(X_list)//num_classes} per class)")

In [ ]:
# -----------------------------------------------------------------------------
# 5. Tensor Conversion and DataLoader Creation
# -----------------------------------------------------------------------------
# Convert lists to NumPy arrays
X_np = np.stack(X_list)  # Shape: [N, D, H, W]
y_np = np.array(Y_list)

# Convert to PyTorch tensors: add channel dimension and one-hot encode labels
X_dots = torch.tensor(X_np).unsqueeze(1).float()                   # [N,1,D,H,W]
y_indices = torch.tensor(y_np)
y_dots = F.one_hot(y_indices, num_classes).float()                # [N, num_classes]

print(f"Tensor shapes -> X: {X_dots.shape}, y: {y_dots.shape}")

# Create inference DataLoader (no shuffling)
test_loader = DataLoader(
    TensorDataset(X_dots, y_dots),
    batch_size=hyperparams['batch_size'],
    shuffle=False
)

In [ ]:
# -----------------------------------------------------------------------------
# 6. Model Loading and Setup
# -----------------------------------------------------------------------------
# Load checkpoint dictionary
checkpoint = torch.load("classifier3d_model.pth", map_location=device)
# Extract architecture parameters
stages = checkpoint['stages']
pooling_configs = checkpoint['pooling_configs']
num_classes_ckpt = checkpoint['num_classes']

# Initialize and load the 3D CNN model
model_3D = Classifier3D(
    stages=stages,
    pooling_configs=pooling_configs,
    num_classes=num_classes_ckpt
).to(device)
model_3D.load_state_dict(checkpoint['model_state_dict'])
model_3D.eval()  # Inference mode

print("Classifier3D model loaded and ready for inference.")

In [ ]:
# -----------------------------------------------------------------------------
# 7. Inference and Confusion Matrix Plotting
# -----------------------------------------------------------------------------
with torch.no_grad():
    preds, trues = [], []
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model_3D(X_batch)
        _, predicted = torch.max(outputs, dim=1)
        preds.extend(predicted.cpu().numpy())
        trues.extend(torch.argmax(y_batch, dim=1).cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(trues, preds)

# Plot the confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=knots, yticklabels=knots
)
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('4-Foil Classifier3D Confusion Matrix')
plt.show()
